# Candidate Generation Check

This notebook validates phase 01 across **all traced floor plans** through the canonical resolver API, then inspects both the full eligible candidate set and the final thinned optimization-facing candidate set for each layout.


In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from src.common.floorplan import OPEN_CELL, SOLID_CELL
from src.common.floorplan_loader import load_traced_floorplan
from src.planner._shared.bootstrap import (
    find_repo_root,
    get_traced_floorplan_path,
    list_traced_floorplan_names,
)
from src.planner._shared.config import PlannerConfig
from src.planner.phase01_candidate_generation import (
    generate_candidate_generation_artifacts,
    resolve_candidate_generation_artifacts,
    validate_candidate_generation_artifacts,
)
from src.planner.presentation import plot_candidate_set_comparison


In [ ]:
repo_root = find_repo_root()
floorplan_names = list_traced_floorplan_names(repo_root=repo_root)
results = []


def has_solid_4_neighbor(grid: np.ndarray, row: int, col: int) -> bool:
    """Return whether one open cell touches a solid cell in the 4-neighborhood."""

    for dr, dc in ((-1, 0), (0, 1), (1, 0), (0, -1)):
        rr = row + dr
        cc = col + dc
        if 0 <= rr < grid.shape[0] and 0 <= cc < grid.shape[1] and grid[rr, cc] == SOLID_CELL:
            return True
    return False


for floorplan_name in floorplan_names:
    config = PlannerConfig(floorplan_name=floorplan_name)
    floorplan = load_traced_floorplan(
        get_traced_floorplan_path(floorplan_name, repo_root=repo_root)
    )
    artifacts = resolve_candidate_generation_artifacts(
        floorplan,
        config,
        repo_root=repo_root,
    )
    validate_candidate_generation_artifacts(
        floorplan,
        artifacts,
        config=config,
    )

    rerun = generate_candidate_generation_artifacts(floorplan, config)
    for left, right in (
        (artifacts.open_cell_indices, rerun.open_cell_indices),
        (artifacts.open_cell_coords_rc, rerun.open_cell_coords_rc),
        (artifacts.eligible_candidate_cell_indices, rerun.eligible_candidate_cell_indices),
        (artifacts.eligible_candidate_cell_coords_rc, rerun.eligible_candidate_cell_coords_rc),
        (artifacts.eligible_candidate_boundary_flags, rerun.eligible_candidate_boundary_flags),
        (artifacts.candidate_cell_indices, rerun.candidate_cell_indices),
        (artifacts.candidate_cell_coords_rc, rerun.candidate_cell_coords_rc),
        (artifacts.candidate_boundary_flags, rerun.candidate_boundary_flags),
        (artifacts.candidate_exception_flags, rerun.candidate_exception_flags),
    ):
        assert np.array_equal(left, right)

    grid = floorplan.grid
    eligible_index_set = set(artifacts.eligible_candidate_cell_indices.tolist())
    for row, col in artifacts.eligible_candidate_cell_coords_rc:
        assert grid[row, col] == OPEN_CELL
        assert has_solid_4_neighbor(grid, int(row), int(col))

    for flat_index, (row, col) in zip(
        artifacts.open_cell_indices,
        artifacts.open_cell_coords_rc,
        strict=True,
    ):
        if int(flat_index) not in eligible_index_set:
            assert not has_solid_4_neighbor(grid, int(row), int(col))

    eligible_unique_flags, eligible_counts = np.unique(
        artifacts.eligible_candidate_boundary_flags,
        return_counts=True,
    )
    final_unique_flags, final_counts = np.unique(
        artifacts.candidate_boundary_flags,
        return_counts=True,
    )
    exception_unique_flags, exception_counts = np.unique(
        artifacts.candidate_exception_flags,
        return_counts=True,
    )

    results.append(
        {
            "floorplan": floorplan,
            "artifacts": artifacts,
            "summary": {
                "floorplan_name": floorplan.name,
                "grid_shape": floorplan.shape,
                "open_cell_count": int(len(artifacts.open_cell_indices)),
                "eligible_candidate_count": int(len(artifacts.eligible_candidate_cell_indices)),
                "candidate_cell_count": int(len(artifacts.candidate_cell_indices)),
                "thinning_ratio": float(len(artifacts.candidate_cell_indices) / len(artifacts.eligible_candidate_cell_indices)) if len(artifacts.eligible_candidate_cell_indices) else 0.0,
                "eligible_boundary_flag_histogram": {int(flag): int(count) for flag, count in zip(eligible_unique_flags, eligible_counts, strict=True)},
                "candidate_boundary_flag_histogram": {int(flag): int(count) for flag, count in zip(final_unique_flags, final_counts, strict=True)},
                "candidate_exception_flag_histogram": {int(flag): int(count) for flag, count in zip(exception_unique_flags, exception_counts, strict=True)},
                "non_exception_candidate_count": int(np.count_nonzero(artifacts.candidate_exception_flags == 0)),
                "exception_candidate_count": int(np.count_nonzero(artifacts.candidate_exception_flags != 0)),
            },
        }
    )

summary_rows = [item["summary"] for item in results]
summary_rows


In [ ]:
for item in results:
    figure, axes = plt.subplots(1, 2, figsize=(10, 4))
    plot_candidate_set_comparison(
        item["floorplan"],
        item["artifacts"],
        axes=axes,
        eligible_title=f"{item['floorplan'].name} eligible candidates",
        final_title=f"{item['floorplan'].name} optimization candidates",
    )
    plt.tight_layout()
    plt.show()
